In [1]:
import spikeinterface.full as si
import probeinterface as pi
from pathlib import Path
import warnings


In [2]:
data_path = '/home/avadher/Data/Adrian/H7113-250604/'
sorter = "mountainsort5"

# Set paths 
data_path = Path(data_path)
session_name = data_path.name
spikesorting_folder_name = f"{session_name}_sorting_{sorter}"
spikesorting_path = data_path / spikesorting_folder_name
analyzer_folder_name = f"{session_name}_analyzer_{sorter}"
analyzer_path = data_path / analyzer_folder_name

# Set the number for cores for parallel processing 
si.set_global_job_kwargs(n_jobs=12) 

# Ignore annoying warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [12]:
# Load recording and probe
recording = si.read_openephys(data_path, stream_id='0')
probe_manufacturer = "cambridgeneurotech"
probe_name = "ASSY-156-H7" 
probe = pi.get_probe(manufacturer=probe_manufacturer, probe_name=probe_name) # pi.list_all_probes() to list all probes 
probe.set_device_channel_indices(list(range(64)))
recording = recording.set_probe(probe, group_mode='by_probe')


In [13]:
# Default channel names are CH1, CH2 etc

print(recording.get_property("channel_name"))


['CH1' 'CH2' 'CH3' 'CH4' 'CH5' 'CH6' 'CH7' 'CH8' 'CH9' 'CH10' 'CH11'
 'CH12' 'CH13' 'CH14' 'CH15' 'CH16' 'CH17' 'CH18' 'CH19' 'CH20' 'CH21'
 'CH22' 'CH23' 'CH24' 'CH25' 'CH26' 'CH27' 'CH28' 'CH29' 'CH30' 'CH31'
 'CH32' 'CH33' 'CH34' 'CH35' 'CH36' 'CH37' 'CH38' 'CH39' 'CH40' 'CH41'
 'CH42' 'CH43' 'CH44' 'CH45' 'CH46' 'CH47' 'CH48' 'CH49' 'CH50' 'CH51'
 'CH52' 'CH53' 'CH54' 'CH55' 'CH56' 'CH57' 'CH58' 'CH59' 'CH60' 'CH61'
 'CH62' 'CH63' 'CH64']


In [14]:
# Change channel names to 0-based integers

channel_ids = recording.get_channel_ids()
new_channel_ids = list(range(len(channel_ids)))
recording = recording.rename_channels(new_channel_ids)
recording.set_property('channel_name', new_channel_ids) # this is used by NeuroConv when creating ElectrodeTable (column 'channel_name')
print(recording.get_property("channel_name"))


[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63]


In [6]:
# Run spike sorting

sorting = si.run_sorter(
    recording=recording, 
    sorter_name=sorter, 
    remove_existing_folder=True, 
    folder=spikesorting_path)


write_binary_recording 
engine=process - n_jobs=12 - samples_per_chunk=30,000 - chunk_memory=7.32 MiB - total_memory=87.89 MiB - chunk_duration=1.00s
Using training recording of duration 300 sec with the sampling mode uniform
*** MS5 Elapsed time for SCHEME2 get_sampled_recording_for_training: 0.674 seconds ***
Running phase 1 sorting
Number of channels: 64
Number of timepoints: 9000000
Sampling frequency: 30000.0 Hz
Channel 0: [226. 425.]
Channel 1: [228. 225.]
Channel 2: [225. 525.]
Channel 3: [223.5 675. ]
Channel 4: [254. 400.]
Channel 5: [252.5 250. ]
Channel 6: [256.5 650. ]
Channel 7: [250.   0.]
Channel 8: [255. 500.]
Channel 9: [251.5 150. ]
Channel 10: [257.5 750. ]
Channel 11: [222.5 775. ]
Channel 12: [229. 125.]
Channel 13: [224. 625.]
Channel 14: [225.5 475. ]
Channel 15: [227.5 275. ]
Channel 16: [  3. 300.]
Channel 17: [  5. 500.]
Channel 18: [  6.5 650. ]
Channel 19: [  1.5 150. ]
Channel 20: [-20.  25.]
Channel 21: [0. 0.]
Channel 22: [-22.5 275. ]
Channel 23: [-26. 6

In [10]:
## Compute analyzer
analyzer = si.create_sorting_analyzer(
    recording=recording,
    sorting=sorting,
    format="binary_folder",
    folder=analyzer_path,
    overwrite=True,
)

#Channel_name is still 0-based integers:
print(analyzer.recording.get_property("channel_name"))

estimate_sparsity (workers: 12 processes):   0%|          | 0/649 [00:00<?, ?it/s]

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63]


In [11]:
# When loaded from analyzer_path, channel_name property reverts to default

analyzer = si.load_sorting_analyzer(analyzer_path)
print(analyzer.recording.get_property("channel_name"))


['CH1' 'CH2' 'CH3' 'CH4' 'CH5' 'CH6' 'CH7' 'CH8' 'CH9' 'CH10' 'CH11'
 'CH12' 'CH13' 'CH14' 'CH15' 'CH16' 'CH17' 'CH18' 'CH19' 'CH20' 'CH21'
 'CH22' 'CH23' 'CH24' 'CH25' 'CH26' 'CH27' 'CH28' 'CH29' 'CH30' 'CH31'
 'CH32' 'CH33' 'CH34' 'CH35' 'CH36' 'CH37' 'CH38' 'CH39' 'CH40' 'CH41'
 'CH42' 'CH43' 'CH44' 'CH45' 'CH46' 'CH47' 'CH48' 'CH49' 'CH50' 'CH51'
 'CH52' 'CH53' 'CH54' 'CH55' 'CH56' 'CH57' 'CH58' 'CH59' 'CH60' 'CH61'
 'CH62' 'CH63' 'CH64']
